# Message Scale and Source–Stance Distribution Probe

- **Project:** Opinion-leader interpersonal diffusion model
- **Submodel ID:** SM-OL-WEIGHT-01
- **Framework link:** opleader message aggregation
- **Probe type:** Deterministic mechanism probe
- **Date:** 2026-09-07
- **Status:** Runnable

## 1. Question, decision, and claim boundary

- **Primary question:** How do the total volume and realized source–stance composition of messages received in one round affect a recipient's Beta-belief mean and concentration under different opinion-leader evidence multipliers?
- **Decision supported:** Check whether the source-only aggregation rule behaves transparently enough to retain for the connected framework.
- **Scale:** The total number of received messages in the supplied exposure set.
- **Distribution:** The realized joint composition across ordinary-support, ordinary-oppose, leader-support, and leader-oppose messages. It is not a stochastic sampling distribution.
- **Highest intended claim:** Behavior of the isolated aggregation mechanism coupled directly to the existing Beta update under fixed synthetic inputs.

## 2. Provenance and boundary contract

Two-step-flow research motivates examining source-dependent interpersonal influence, but it does not specify the linear multiplier or any value used here. The source-only weighting rule is a project operationalization. All multiplier values, message scales, distributions, and the fixed Beta prior in this notebook are illustrative rather than calibrated.

Every case supplies an exact one-round exposure set. Message origination, selection, network topology, recipient-role differences, multi-round propagation, and all feedbacks are omitted. There is no randomness or seed. Holding the prior fixed isolates how message scale and joint composition enter the existing update.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "src" / "opinion_model").is_dir()
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from opinion_model.shared import aggregate_messages, propose_opinion_update
from opinion_model.core import (
    AgentState,
    AggregationContext,
    BetaBelief,
    Exposure,
    Message,
)
from opinion_model.opleader import OpinionLeaderMessageAggregation

sns.set_theme(style="whitegrid")

RUN = {
    "submodel_id": "SM-OL-WEIGHT-01",
    "submodel_version": "0.3",
    "boundary_scenario": "fixed one-round synthetic exposures",
    "prior": {"a": 2.0, "b": 2.0},
    "base_evidence_weight": 0.1,
    "message_scales": (4, 20, 40),
    "leader_multipliers": (1.0, 2.0, 4.0),
    "active_feedbacks": (),
    "omitted_processes": (
        "origination",
        "selection",
        "network",
        "recipient-role effects",
        "multi-round propagation",
    ),
}
RUN

## 3. Scenario construction

All producers are unique, consistent with the current one-message-per-originator rule. The four shares in each named distribution sum to one and convert exactly to integer counts at every selected scale.

The distributions include aligned stances, neutral balance, different source shares, and two conflicts with identical overall support and opposition but reversed source–stance allocation.

In [ ]:
CONSUMER_ID = 10_000
CATEGORY_ORDER = (
    "ordinary_support",
    "ordinary_oppose",
    "leader_support",
    "leader_oppose",
)
CATEGORY_PROPERTIES = {
    "ordinary_support": {"is_leader": False, "stance": 1},
    "ordinary_oppose": {"is_leader": False, "stance": -1},
    "leader_support": {"is_leader": True, "stance": 1},
    "leader_oppose": {"is_leader": True, "stance": -1},
}
DISTRIBUTIONS = {
    "ordinary support only": {
        "ordinary_support": 1.0,
        "ordinary_oppose": 0.0,
        "leader_support": 0.0,
        "leader_oppose": 0.0,
    },
    "leader support only": {
        "ordinary_support": 0.0,
        "ordinary_oppose": 0.0,
        "leader_support": 1.0,
        "leader_oppose": 0.0,
    },
    "mixed support": {
        "ordinary_support": 0.5,
        "ordinary_oppose": 0.0,
        "leader_support": 0.5,
        "leader_oppose": 0.0,
    },
    "mixed opposition": {
        "ordinary_support": 0.0,
        "ordinary_oppose": 0.5,
        "leader_support": 0.0,
        "leader_oppose": 0.5,
    },
    "fully balanced": {
        "ordinary_support": 0.25,
        "ordinary_oppose": 0.25,
        "leader_support": 0.25,
        "leader_oppose": 0.25,
    },
    "leader supports / ordinary opposes": {
        "ordinary_support": 0.0,
        "ordinary_oppose": 0.5,
        "leader_support": 0.5,
        "leader_oppose": 0.0,
    },
    "ordinary supports / leader opposes": {
        "ordinary_support": 0.5,
        "ordinary_oppose": 0.0,
        "leader_support": 0.0,
        "leader_oppose": 0.5,
    },
}


def exact_counts(message_scale, distribution):
    shares = DISTRIBUTIONS[distribution]
    counts = {
        category: int(message_scale * shares[category])
        for category in CATEGORY_ORDER
    }
    if any(
        not np.isclose(counts[category], message_scale * shares[category])
        for category in CATEGORY_ORDER
    ):
        raise ValueError("Message scale does not produce exact integer counts.")
    if sum(counts.values()) != message_scale:
        raise ValueError("Joint message counts must sum to message_scale.")
    return counts


def make_exposures(counts):
    exposures = []
    leader_ids = set()
    producer_id = 0
    for category in CATEGORY_ORDER:
        properties = CATEGORY_PROPERTIES[category]
        for within_category_index in range(counts[category]):
            if properties["is_leader"]:
                leader_ids.add(producer_id)
            exposures.append(
                Exposure(
                    round_index=1,
                    consumer_id=CONSUMER_ID,
                    message=Message(
                        message_id=(
                            f"r1:{category}:{within_category_index}"
                        ),
                        round_index=1,
                        producer_id=producer_id,
                        stance=properties["stance"],
                    ),
                )
            )
            producer_id += 1
    return tuple(exposures), frozenset(leader_ids)


def evaluate_case(message_scale, distribution, leader_multiplier):
    counts = exact_counts(message_scale, distribution)
    exposures, leader_ids = make_exposures(counts)
    aggregator = OpinionLeaderMessageAggregation(
        leader_ids=leader_ids,
        leader_evidence_multiplier=leader_multiplier,
    )
    evidence = aggregator(
        exposures,
        AggregationContext(RUN["base_evidence_weight"]),
    )
    before = AgentState(BetaBelief(**RUN["prior"]))
    after = propose_opinion_update(before, evidence)
    return {
        "message_scale": message_scale,
        "distribution": distribution,
        "leader_multiplier": leader_multiplier,
        **counts,
        "raw_support": evidence.n_support,
        "raw_oppose": evidence.n_oppose,
        "weighted_support": evidence.weighted_support,
        "weighted_oppose": evidence.weighted_oppose,
        "a_before": before.belief.a,
        "b_before": before.belief.b,
        "a_after": after.belief.a,
        "b_after": after.belief.b,
        "mean_before": before.belief.mean,
        "mean_after": after.belief.mean,
        "mean_change": after.belief.mean - before.belief.mean,
        "concentration_before": before.belief.concentration,
        "concentration_after": after.belief.concentration,
        "concentration_change": (
            after.belief.concentration - before.belief.concentration
        ),
    }

## 4. Deterministic and boundary checks

The checks below establish the null, the directional effect of source–stance allocation, and the exact link between weighted evidence and Beta concentration before the factorial comparison.

In [ ]:
context = AggregationContext(RUN["base_evidence_weight"])
prior = AgentState(BetaBelief(**RUN["prior"]))
empty_aggregator = OpinionLeaderMessageAggregation(
    leader_ids=frozenset(),
    leader_evidence_multiplier=2.0,
)
empty_evidence = empty_aggregator((), context)
empty_after = propose_opinion_update(prior, empty_evidence)
assert empty_after == prior

conflict_name = "leader supports / ordinary opposes"
reverse_conflict_name = "ordinary supports / leader opposes"
conflict_counts = exact_counts(4, conflict_name)
conflict_exposures, conflict_leaders = make_exposures(conflict_counts)
unit_aggregator = OpinionLeaderMessageAggregation(
    leader_ids=conflict_leaders,
    leader_evidence_multiplier=1.0,
)
assert unit_aggregator(
    conflict_exposures,
    context,
) == aggregate_messages(conflict_exposures, context)

null_conflict = evaluate_case(4, conflict_name, 1.0)
weighted_conflict = evaluate_case(4, conflict_name, 4.0)
weighted_reverse = evaluate_case(4, reverse_conflict_name, 4.0)
assert np.isclose(null_conflict["mean_after"], 0.5)
assert weighted_conflict["mean_after"] > 0.5
assert weighted_reverse["mean_after"] < 0.5
assert np.isclose(
    weighted_conflict["concentration_change"],
    (
        weighted_conflict["weighted_support"]
        + weighted_conflict["weighted_oppose"]
    ),
)

checks = pd.DataFrame(
    [null_conflict, weighted_conflict, weighted_reverse]
)[
    [
        "distribution",
        "leader_multiplier",
        "raw_support",
        "raw_oppose",
        "weighted_support",
        "weighted_oppose",
        "mean_after",
        "concentration_after",
    ]
]
checks.round(4)

## 5. Factorial comparison

The comparison crosses the three selected message scales, seven realized joint distributions, and three leader multipliers. The recipient's prior and the base evidence weight remain fixed in every row.

In [ ]:
rows = []
for message_scale in RUN["message_scales"]:
    for distribution in DISTRIBUTIONS:
        for leader_multiplier in RUN["leader_multipliers"]:
            rows.append(
                evaluate_case(
                    message_scale,
                    distribution,
                    leader_multiplier,
                )
            )

results = (
    pd.DataFrame(rows)
    .sort_values(
        ["leader_multiplier", "distribution", "message_scale"]
    )
    .reset_index(drop=True)
)

assert len(results) == (
    len(RUN["message_scales"])
    * len(DISTRIBUTIONS)
    * len(RUN["leader_multipliers"])
)
assert np.allclose(
    results["concentration_change"],
    results["weighted_support"] + results["weighted_oppose"],
)

display_columns = [
    "message_scale",
    "distribution",
    "leader_multiplier",
    "raw_support",
    "raw_oppose",
    "weighted_support",
    "weighted_oppose",
    "a_after",
    "b_after",
    "mean_after",
    "concentration_after",
]
results[display_columns].round(4)

In [ ]:
distribution_order = list(DISTRIBUTIONS)
scale_order = list(RUN["message_scales"])
multipliers = list(RUN["leader_multipliers"])

fig, axes = plt.subplots(
    1,
    len(multipliers),
    figsize=(15, 6),
    sharey=True,
)
for axis, multiplier in zip(axes, multipliers):
    pivot = (
        results.loc[results["leader_multiplier"] == multiplier]
        .pivot(
            index="distribution",
            columns="message_scale",
            values="mean_after",
        )
        .reindex(index=distribution_order, columns=scale_order)
    )
    sns.heatmap(
        pivot,
        annot=True,
        fmt=".3f",
        cmap="coolwarm",
        center=0.5,
        vmin=0.0,
        vmax=1.0,
        cbar=multiplier == multipliers[-1],
        ax=axis,
    )
    axis.set_title(f"Leader multiplier = {multiplier:g}")
    axis.set_xlabel("Message scale")
    axis.set_ylabel("Distribution" if multiplier == multipliers[0] else "")
fig.suptitle("Posterior Beta mean", y=1.02)
fig.tight_layout()
plt.show()

concentration_min = results["concentration_after"].min()
concentration_max = results["concentration_after"].max()
fig, axes = plt.subplots(
    1,
    len(multipliers),
    figsize=(15, 6),
    sharey=True,
)
for axis, multiplier in zip(axes, multipliers):
    pivot = (
        results.loc[results["leader_multiplier"] == multiplier]
        .pivot(
            index="distribution",
            columns="message_scale",
            values="concentration_after",
        )
        .reindex(index=distribution_order, columns=scale_order)
    )
    sns.heatmap(
        pivot,
        annot=True,
        fmt=".1f",
        cmap="viridis",
        vmin=concentration_min,
        vmax=concentration_max,
        cbar=multiplier == multipliers[-1],
        ax=axis,
    )
    axis.set_title(f"Leader multiplier = {multiplier:g}")
    axis.set_xlabel("Message scale")
    axis.set_ylabel("Distribution" if multiplier == multipliers[0] else "")
fig.suptitle("Posterior Beta concentration", y=1.02)
fig.tight_layout()
plt.show()

## 6. Conditional interpretation

The following summary reports selected deterministic outcomes. These results describe the specified linear evidence rule; they do not validate a multiplier, message distribution, or population-level effect.

In [ ]:
def result_at(message_scale, distribution, leader_multiplier):
    match = results[
        (results["message_scale"] == message_scale)
        & (results["distribution"] == distribution)
        & (results["leader_multiplier"] == leader_multiplier)
    ]
    assert len(match) == 1
    return match.iloc[0]


ordinary_support = result_at(40, "ordinary support only", 4.0)
leader_support = result_at(40, "leader support only", 4.0)
balanced = result_at(40, "fully balanced", 4.0)
leader_conflict = result_at(40, conflict_name, 4.0)
ordinary_conflict = result_at(40, reverse_conflict_name, 4.0)
null_conflict_large = result_at(40, conflict_name, 1.0)

display(
    Markdown(
        f"""
- **Scale effect:** Forty leader-support messages with multiplier 4 move the posterior mean to **{leader_support['mean_after']:.3f}** and concentration to **{leader_support['concentration_after']:.1f}**.
- **Source-share effect:** At the same scale and stance, forty ordinary-support messages end at mean **{ordinary_support['mean_after']:.3f}**, compared with **{leader_support['mean_after']:.3f}** for leader-support messages.
- **Homogeneous null:** With equal total support and opposition, the conflict case remains at mean **{null_conflict_large['mean_after']:.3f}** when the leader multiplier is 1.
- **Source–stance allocation:** With the same overall support and opposition counts but multiplier 4, leader support ends at mean **{leader_conflict['mean_after']:.3f}**, while leader opposition ends at **{ordinary_conflict['mean_after']:.3f}**.
- **Balanced evidence:** The fully balanced distribution remains at mean **{balanced['mean_after']:.3f}** while concentration rises to **{balanced['concentration_after']:.1f}**.
- **Highest completed level:** Source-only aggregation coupled to the existing Beta update under fixed one-round synthetic exposures.
"""
    )
)

## 7. Disposition

The probe makes message scale, stance composition, source composition, and source–stance allocation observable without introducing a second aggregation implementation. It supports retaining the source-only mechanism as a transparent working representation.

The multiplier values and synthetic distributions remain illustrative. The probe does not establish empirical credibility effects, realistic exposure volumes, recipient heterogeneity, network outcomes, or multi-round dynamics. Those claims require evidence and experiments at their corresponding coupling levels.